### Feature Selection:
In this we only select features which are important for model. In other words which have higher correlaton with the target variable.

In [1]:
import pandas as pd
import numpy as np

In [6]:
df = pd.read_csv("home_prices.csv")
df.head()

,area_sqr_ft,bedrooms,color,price_lakhs
0,3774,2,Red,216
1,1460,3,Gray,88
2,1894,4,Gray,147
3,1730,2,Blue,84
4,1695,1,Blue,77


As the color is categorical column also in nominal form so we will apply one hot encoding to it. So that we can use corr method on it to calculate correlation matrix.

In [7]:
df = pd.get_dummies(data=df, columns=['color'], drop_first=True)
df.head()

,area_sqr_ft,bedrooms,price_lakhs,color_Gray,color_Green,color_Red,color_White,color_Yellow
0,3774,2,216,False,False,True,False,False
1,1460,3,88,True,False,False,False,False
2,1894,4,147,True,False,False,False,False
3,1730,2,84,False,False,False,False,False
4,1695,1,77,False,False,False,False,False


Now let's calculte correlation matrix to find features which are not important enough for our model so we can replace them.

In [9]:
corr = df.corr()
corr

,area_sqr_ft,bedrooms,price_lakhs,color_Gray,color_Green,color_Red,color_White,color_Yellow
area_sqr_ft,1.000000,0.185810,0.945365,-0.068944,-0.032012,0.059055,0.063827,-0.037819
bedrooms,0.185810,1.000000,0.439445,0.040882,-0.120207,-0.004177,-0.023676,0.015286
price_lakhs,0.945365,0.439445,1.000000,-0.040565,-0.041959,0.045803,0.051122,-0.046673
color_Gray,-0.068944,0.040882,-0.040565,1.000000,-0.214409,-0.230990,-0.205931,-0.217205
color_Green,-0.032012,-0.120207,-0.041959,-0.214409,1.000000,-0.190117,-0.169493,-0.178771
color_Red,0.059055,-0.004177,0.045803,-0.230990,-0.190117,1.000000,-0.182600,-0.192596
color_White,0.063827,-0.023676,0.051122,-0.205931,-0.169493,-0.182600,1.000000,-0.171703
color_Yellow,-0.037819,0.015286,-0.046673,-0.217205,-0.178771,-0.192596,-0.171703,1.000000


This is complete correlation matrix but we are mainly interested on target variable which is price_lakhs corr values.

In [10]:
corr['price_lakhs']

area_sqr_ft     0.945365
bedrooms        0.439445
price_lakhs     1.000000
color_Gray     -0.040565
color_Green    -0.041959
color_Red       0.045803
color_White     0.051122
color_Yellow   -0.046673
Name: price_lakhs, dtype: float64

As u can see that color column corr is so small which is why it's not an important feature to have we can simply drop them

In reality we can have many features in our dataset so in that case we decide threshold for correlation(for ex 0.2). So Let's get the abs values now so we can apply threshold limit to it. 

In [13]:
# abs values
corr = abs(corr)
corr_target = corr['price_lakhs']
corr_target

area_sqr_ft     0.945365
bedrooms        0.439445
price_lakhs     1.000000
color_Gray      0.040565
color_Green     0.041959
color_Red       0.045803
color_White     0.051122
color_Yellow    0.046673
Name: price_lakhs, dtype: float64

In [17]:
threshold = 0.2

imp_features = corr_target[corr_target > 0.2].index.drop('price_lakhs')  # We can get the indexs of those columns and drop the target column 
imp_features

Index(['area_sqr_ft', 'bedrooms'], dtype='object')

In [19]:
X = df[imp_features]
y = df['price_lakhs']

In [21]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# model training
model = LinearRegression()
model.fit(X_train, y_train)

# Prediciton
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2s = r2_score(y_test, y_pred)
print(f"MSE:{mse} , R2: {r2s}")

MSE:76.63332198278805 , R2: 0.9689466488379601


Scenarios where you should not use correlation for feature selection:
* Non Linear relationships
* Outliers
* Categorical Variables
* Correlation Vs Causation